In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

from prophet import Prophet

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 5)

## Part 1 — Ingestion & Time-Series Preparation

The dataset is loaded directly from the mandatory raw CSV source. The raw column names are standardized to `Month` and `Production`, then `Month` is converted to a monthly-start (`MS`) DatetimeIndex.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/plotly/datasets/master/monthly-milk-production-pounds.csv"

df = pd.read_csv(DATA_URL)

# Standardize the two required columns
df.columns = ["Month", "Production"]

# Convert Month to datetime and sort chronologically
df["Month"] = pd.to_datetime(df["Month"])
df = df.sort_values("Month").set_index("Month")

# Enforce monthly-start frequency
df.index = pd.DatetimeIndex(df.index).to_period("M").to_timestamp(how="start")
df = df.asfreq("MS")

print("Shape:", df.shape)
print("Start:", df.index.min())
print("End:", df.index.max())
print("\nMissing values:")
print(df.isna().sum())

df.head()

## Part 2 — Visualize the Complete Historical Series

The historical series is expected to show three systematic components:

- **Trend:** a persistent upward movement in milk production.
- **Annual seasonality:** a repeating pattern every 12 months.
- **Irregular noise:** short-term deviations around the trend and seasonal pattern.

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(df.index, df["Production"], linewidth=1.8)

plt.title("Monthly Milk Production (1962–1975)")
plt.xlabel("Month")
plt.ylabel("Production (pounds per cow)")
plt.grid(True)

plt.show()

## Part 3 — Chronological 80/20 Train-Test Split

For time-series forecasting, observations must remain in chronological order. Random shuffling is avoided because it would allow information from the future to influence model evaluation.

In [ ]:
train_size = int(len(df) * 0.8)

train = df.iloc[:train_size].copy()
test = df.iloc[train_size:].copy()

print("Total observations:", len(df))
print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Train period:", train.index.min(), "to", train.index.max())
print("Test period:", test.index.min(), "to", test.index.max())

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(train.index, train["Production"], label="Training Data")
plt.plot(test.index, test["Production"], label="Test Data")

plt.title("Chronological 80/20 Train-Test Split")
plt.xlabel("Month")
plt.ylabel("Production")
plt.legend()
plt.grid(True)

plt.show()

## Part 4 — Evaluation Helper

The models are evaluated using:

- **MAE:** Mean Absolute Error
- **RMSE:** Root Mean Squared Error

Lower values indicate better out-of-sample forecasting performance.

In [ ]:
def evaluate_model(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    return mae, rmse

## Model 1 — Naive Forecast

The Naive model projects the final observed training value across the entire test horizon:

$$\hat{Y}_{t+h} = Y_T$$

In [ ]:
last_value = train["Production"].iloc[-1]

naive_forecast = np.repeat(
    last_value,
    len(test)
)

naive_mae, naive_rmse = evaluate_model(
    test["Production"],
    naive_forecast
)

print("Naive Model")
print("MAE :", naive_mae)
print("RMSE:", naive_rmse)

## Model 2 — Autoregressive AR(2)

The AR(2) model uses the previous two observations:

$$Y_t = c + \phi_1Y_{t-1} + \phi_2Y_{t-2} + \epsilon_t$$

In [ ]:
ar_model = AutoReg(
    train["Production"],
    lags=2,
    trend="c"
).fit()

ar_forecast = ar_model.predict(
    start=len(train),
    end=len(train) + len(test) - 1
)

ar_mae, ar_rmse = evaluate_model(
    test["Production"],
    ar_forecast
)

print("AR(2)")
print("MAE :", ar_mae)
print("RMSE:", ar_rmse)

## Model 3 — Moving Average MA(2)

The MA(2) model is implemented with:

$$ARIMA(0,0,2)$$

It models the current value using the current error and the two previous error terms.

In [ ]:
ma_model = ARIMA(
    train["Production"],
    order=(0, 0, 2)
).fit()

ma_forecast = ma_model.forecast(
    steps=len(test)
)

ma_mae, ma_rmse = evaluate_model(
    test["Production"],
    ma_forecast
)

print("MA(2)")
print("MAE :", ma_mae)
print("RMSE:", ma_rmse)

## Model 4 — ARMA(2,2)

ARMA combines autoregressive and moving-average components.

In `statsmodels`, ARMA(2,2) is represented as:

$$ARIMA(2,0,2)$$

In [ ]:
arma_model = ARIMA(
    train["Production"],
    order=(2, 0, 2)
).fit()

arma_forecast = arma_model.forecast(
    steps=len(test)
)

arma_mae, arma_rmse = evaluate_model(
    test["Production"],
    arma_forecast
)

print("ARMA(2,2)")
print("MAE :", arma_mae)
print("RMSE:", arma_rmse)

## Model 5 — ARIMA(2,1,2)

The first-order differencing term (`d=1`) is used to account for the non-stationary upward trend:

$$ARIMA(2,1,2)$$

In [ ]:
arima_model = ARIMA(
    train["Production"],
    order=(2, 1, 2)
).fit()

arima_forecast = arima_model.forecast(
    steps=len(test)
)

arima_mae, arima_rmse = evaluate_model(
    test["Production"],
    arima_forecast
)

print("ARIMA(2,1,2)")
print("MAE :", arima_mae)
print("RMSE:", arima_rmse)

## Model 6 — Seasonal SARIMA

Because the observations are monthly and the series has annual seasonality, the seasonal period is:

$$s=12$$

The required model is:

$$SARIMA(1,1,1)(1,1,1,12)$$

In [ ]:
sarima_model = SARIMAX(
    train["Production"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit()

sarima_forecast = sarima_model.forecast(
    steps=len(test)
)

sarima_mae, sarima_rmse = evaluate_model(
    test["Production"],
    sarima_forecast
)

print("SARIMA(1,1,1)(1,1,1,12)")
print("MAE :", sarima_mae)
print("RMSE:", sarima_rmse)

## Model 7 — Meta Prophet

Prophet requires:

- `ds` = date
- `y` = target

Only yearly seasonality is enabled because the data is monthly and the task specifically requires `yearly_seasonality=True`.

In [ ]:
prophet_train = train.reset_index().rename(
    columns={
        "Month": "ds",
        "Production": "y"
    }
)

prophet_test = test.reset_index().rename(
    columns={
        "Month": "ds",
        "Production": "y"
    }
)

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

prophet_model.fit(prophet_train)

In [ ]:
future = prophet_model.make_future_dataframe(
    periods=len(test),
    freq="MS"
)

forecast = prophet_model.predict(future)

prophet_forecast = (
    forecast[forecast["ds"].isin(prophet_test["ds"])]["yhat"]
    .to_numpy()
)

prophet_mae, prophet_rmse = evaluate_model(
    prophet_test["y"],
    prophet_forecast
)

print("Prophet")
print("MAE :", prophet_mae)
print("RMSE:", prophet_rmse)

## Part 10 — Performance Leaderboard

All seven models are consolidated into one DataFrame and ranked by the lowest test RMSE.

In [ ]:
results = [
    {"Model": "Naive", "MAE": naive_mae, "RMSE": naive_rmse},
    {"Model": "AR(2)", "MAE": ar_mae, "RMSE": ar_rmse},
    {"Model": "MA(2)", "MAE": ma_mae, "RMSE": ma_rmse},
    {"Model": "ARMA(2,2)", "MAE": arma_mae, "RMSE": arma_rmse},
    {"Model": "ARIMA(2,1,2)", "MAE": arima_mae, "RMSE": arima_rmse},
    {"Model": "SARIMA(1,1,1)(1,1,1,12)", "MAE": sarima_mae, "RMSE": sarima_rmse},
    {"Model": "Prophet", "MAE": prophet_mae, "RMSE": prophet_rmse}
]

results_df = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

results_df.index = results_df.index + 1
results_df

## Part 11 — Forecast Comparison Plot

The plot compares the training observations, actual test observations, and forecasts from all seven models.

In [ ]:
forecast_series = {
    "Naive": naive_forecast,
    "AR(2)": ar_forecast,
    "MA(2)": ma_forecast,
    "ARMA(2,2)": arma_forecast,
    "ARIMA(2,1,2)": arima_forecast,
    "SARIMA": sarima_forecast,
    "Prophet": prophet_forecast
}

plt.figure(figsize=(16, 8))

plt.plot(
    train.index,
    train["Production"],
    label="Training Data"
)

plt.plot(
    test.index,
    test["Production"],
    label="Actual Test Data",
    linewidth=3
)

for name, prediction in forecast_series.items():
    plt.plot(
        test.index,
        prediction,
        label=name
    )

plt.title("Monthly Milk Production — Forecast Comparison")
plt.xlabel("Month")
plt.ylabel("Production (pounds per cow)")
plt.legend()
plt.grid(True)

plt.show()

## Part 12 — Autocorrelation Diagnostics

ACF helps identify serial dependence and potential MA/seasonal structure.

PACF helps identify direct autoregressive relationships and can help suggest AR orders.

Because the data is monthly and seasonal, particular attention should be paid to lags around 12 and 24.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_acf(
    train["Production"],
    lags=24,
    ax=ax
)

ax.set_title("Training Series ACF — Lags 0 to 24")
plt.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_pacf(
    train["Production"],
    lags=24,
    ax=ax,
    method="ywm"
)

ax.set_title("Training Series PACF — Lags 0 to 24")
plt.grid(True)
plt.show()

## Part 13 — SARIMA Residual Analysis

Residuals are the differences between observed and fitted values:

$$\epsilon_t = Y_t - \hat{Y}_t$$

A well-specified forecasting model should leave residuals that are approximately uncorrelated and behave like white noise.

In [ ]:
residuals = sarima_model.resid.dropna()

plt.figure(figsize=(14, 5))

plt.plot(residuals)
plt.axhline(
    y=0,
    linestyle="--"
)

plt.title("SARIMA Residuals")
plt.xlabel("Time")
plt.ylabel("Residual")
plt.grid(True)

plt.show()

## Part 14 — Residual ACF

The residual ACF checks whether serial correlation remains after fitting SARIMA.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_acf(
    residuals,
    lags=24,
    ax=ax
)

ax.set_title("SARIMA Residual ACF — Lags 0 to 24")
plt.grid(True)
plt.show()

## Part 15 — Ljung-Box White-Noise Test

The Ljung-Box test evaluates whether a group of autocorrelations is jointly different from zero.

**Null hypothesis (H0):** the residuals are independently distributed / contain no serial autocorrelation up to the tested lag.

Interpretation:

- `p-value > 0.05` → fail to reject H0; residuals are consistent with white noise.
- `p-value <= 0.05` → reject H0; significant autocorrelation remains, suggesting the model has not captured all temporal structure.

In [ ]:
ljung_box = acorr_ljungbox(
    residuals,
    lags=[12, 24],
    return_df=True
)

ljung_box

In [ ]:
for lag, row in ljung_box.iterrows():
    p_value = row["lb_pvalue"]

    if p_value > 0.05:
        conclusion = "Fail to reject H0: residuals are consistent with white noise."
    else:
        conclusion = "Reject H0: significant residual autocorrelation remains."

    print(f"Lag {lag}: p-value = {p_value:.4f}")
    print(conclusion)
    print()

## Final Conclusion

The final model should be selected primarily using the chronological out-of-sample test performance, with **RMSE** as the main ranking metric and **MAE** as a complementary measure.

For this monthly milk-production series, the seasonal structure is especially important. Therefore, the SARIMA and Prophet forecasts should be examined alongside the simpler baselines rather than assuming that a more complex model is automatically better.

The residual diagnostics provide a second validation layer: a strong forecasting model should not only achieve low test errors but should also leave residuals with little remaining autocorrelation.